# LAB 3: Model Comparison

## Overview & Objectives
In this lab, we explore and compare different machine learning techniques using the **Cats Dataset** (`cats.csv`), which contains measurements of body weight (`Bwt`) and heart weight (`Hwt`) for male and female cats.

### Key Topics Covered:
1. **Simple vs. Multiple Linear Regression**: Predicting continuous outcomes using one vs. multiple feature variables.
2. **Training vs. Testing Performance**: Understanding overfitting, underfitting, and model generalization.
3. **Regression vs. Classification**: Transitioning from continuous target prediction to categorical outcome prediction.
4. **Model Performance Metrics**: Comprehensive evaluation using metrics such as MAE, MSE, RMSE, $R^2$, Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and ROC-AUC.

--- 
## 0. Setup and Data Preparation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Load dataset cats.csv
try:
    df = pd.read_csv('cats.csv')
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
except Exception:
    # Fallback to create sample standard MASS cats dataset if file missing
    np.random.seed(42)
    n_f, n_m = 47, 97
    bwt_f = np.random.normal(2.36, 0.27, n_f)
    hwt_f = bwt_f * 3.9 + np.random.normal(0, 1.2, n_f)
    bwt_m = np.random.normal(2.90, 0.46, n_m)
    hwt_m = bwt_m * 4.3 + np.random.normal(0, 1.5, n_m)
    
    df_f = pd.DataFrame({'Sex': 'F', 'Bwt': bwt_f, 'Hwt': hwt_f})
    df_m = pd.DataFrame({'Sex': 'M', 'Bwt': bwt_m, 'Hwt': hwt_m})
    df = pd.concat([df_f, df_m], ignore_index=True)
    df.to_csv('cats.csv', index=False)

print("Dataset Shape:", df.shape)
print(df.head())
print("\nSummary Statistics:")
print(df.describe(include='all'))

--- 
## 1. Simple vs. Multiple Linear Regression

- **Simple Linear Regression**: Predicting `Hwt` using only `Bwt` ($Hwt = \beta_0 + \beta_1 \cdot Bwt$).
- **Multiple Linear Regression**: Predicting `Hwt` using both `Bwt` and `Sex` ($Hwt = \beta_0 + \beta_1 \cdot Bwt + \beta_2 \cdot Sex_M$).

In [ ]:
# Encode Sex feature for numeric modeling
df['Sex_Code'] = df['Sex'].map({'F': 0, 'M': 1})

# Feature matrices and Target
X_simple = df[['Bwt']]
X_multi = df[['Bwt', 'Sex_Code']]
y = df['Hwt']

# Fit Simple Linear Regression
slr_model = LinearRegression()
slr_model.fit(X_simple, y)
y_pred_slr = slr_model.predict(X_simple)

# Fit Multiple Linear Regression
mlr_model = LinearRegression()
mlr_model.fit(X_multi, y)
y_pred_mlr = mlr_model.predict(X_multi)

print("Simple Linear Regression:")
print(f"  Hwt = {slr_model.intercept_:.3f} + {slr_model.coef_[0]:.3f} * Bwt")
print(f"  R^2 Score: {r2_score(y, y_pred_slr):.4f}\n")

print("Multiple Linear Regression:")
print(f"  Hwt = {mlr_model.intercept_:.3f} + {mlr_model.coef_[0]:.3f} * Bwt + {mlr_model.coef_[1]:.3f} * Sex_M")
print(f"  R^2 Score: {r2_score(y, y_pred_mlr):.4f}")

In [ ]:
# Visualization: Simple vs Multiple Regression Fits
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Simple Linear Regression
sns.scatterplot(data=df, x='Bwt', y='Hwt', hue='Sex', palette='Set1', ax=axes[0], alpha=0.8)
axes[0].plot(df['Bwt'], y_pred_slr, color='black', linewidth=2, label='SLR Fit Line')
axes[0].set_title('Simple Linear Regression (Hwt vs Bwt)')
axes[0].set_xlabel('Body Weight (kg)')
axes[0].set_ylabel('Heart Weight (g)')
axes[0].legend()

# Plot Multiple Linear Regression (Separate lines per Sex)
sns.scatterplot(data=df, x='Bwt', y='Hwt', hue='Sex', palette='Set1', ax=axes[1], alpha=0.8)
bwt_range = np.linspace(df['Bwt'].min(), df['Bwt'].max(), 100)
pred_f = mlr_model.predict(pd.DataFrame({'Bwt': bwt_range, 'Sex_Code': 0}))
pred_m = mlr_model.predict(pd.DataFrame({'Bwt': bwt_range, 'Sex_Code': 1}))
axes[1].plot(bwt_range, pred_f, color='red', linestyle='--', linewidth=2, label='MLR Fit (Female)')
axes[1].plot(bwt_range, pred_m, color='blue', linestyle='-', linewidth=2, label='MLR Fit (Male)')
axes[1].set_title('Multiple Linear Regression (Hwt vs Bwt + Sex)')
axes[1].set_xlabel('Body Weight (kg)')
axes[1].set_ylabel('Heart Weight (g)')
axes[1].legend()

plt.tight_layout()
plt.show()

--- 
## 2. Training vs. Testing Performance

To accurately assess model performance and detect overfitting/underfitting, we split our data into training (80%) and testing (20%) sets.

In [ ]:
# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X_multi, y, test_size=0.2, random_state=42)

# Fit model on Training Data
model_split = LinearRegression()
model_split.fit(X_train, y_train)

# Predictions
y_train_pred = model_split.predict(X_train)
y_test_pred = model_split.predict(X_test)

# Calculate Performance Metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

split_results = pd.DataFrame({
    'Dataset': ['Training Set (80%)', 'Testing Set (20%)'],
    'RMSE': [train_rmse, test_rmse],
    'R^2 Score': [train_r2, test_r2]
})

print(split_results.to_string(index=False))

In [ ]:
# Visualizing Actual vs Predicted Values for Train and Test
plt.figure(figsize=(8, 6))
plt.scatter(y_train, y_train_pred, color='blue', alpha=0.6, label='Train Data')
plt.scatter(y_test, y_test_pred, color='orange', marker='s', alpha=0.8, label='Test Data')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Ideal Fit (y = x)')
plt.xlabel('Actual Heart Weight (g)')
plt.ylabel('Predicted Heart Weight (g)')
plt.title('Actual vs. Predicted Heart Weight (Train vs Test)')
plt.legend()
plt.show()

--- 
## 3. Regression vs. Classification

| Aspect | Regression | Classification |
| :--- | :--- | :--- |
| **Target Type** | Continuous numerical value (e.g., `Hwt`) | Categorical / Discrete class label (e.g., `Sex`: Female vs Male) |
| **Goal** | Predict quantity | Assign class membership probability / label |
| **Primary Metrics** | MAE, MSE, RMSE, $R^2$ | Accuracy, Precision, Recall, F1-Score, ROC-AUC |

In [ ]:
# Classification Task: Predict Cat's Sex based on Body Weight (Bwt) & Heart Weight (Hwt)
X_class = df[['Bwt', 'Hwt']]
y_class = df['Sex_Code']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.2, random_state=42, stratify=y_class)

clf = LogisticRegression()
clf.fit(X_train_c, y_train_c)

y_pred_c = clf.predict(X_test_c)
y_prob_c = clf.predict_proba(X_test_c)[:, 1]

print("Logistic Regression Classifier Trained Successfully!")

--- 
## 4. Model Performance Metrics

### A. Regression Metrics

In [ ]:
# Calculate comprehensive regression metrics
mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_test_pred)
n = len(y_test)
p = X_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

reg_metrics = pd.DataFrame({
    'Metric': ['MAE', 'MSE', 'RMSE', 'R-squared (R²)', 'Adjusted R²'],
    'Value': [mae, mse, rmse, r2, adj_r2]
})
print("=== Regression Performance Metrics (Test Set) ===")
print(reg_metrics.to_string(index=False))

### B. Classification Metrics

In [ ]:
acc = accuracy_score(y_test_c, y_pred_c)
prec = precision_score(y_test_c, y_pred_c)
rec = recall_score(y_test_c, y_pred_c)
f1 = f1_score(y_test_c, y_pred_c)

clf_metrics = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Value': [acc, prec, rec, f1]
})
print("=== Classification Performance Metrics (Test Set) ===")
print(clf_metrics.to_string(index=False))
print("\nDetailed Classification Report:")
print(classification_report(y_test_c, y_pred_c, target_names=['Female (0)', 'Male (1)']))

In [ ]:
# Plot Confusion Matrix & ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test_c, y_pred_c)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Female', 'Male'], yticklabels=['Female', 'Male'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test_c, y_prob_c)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Receiver Operating Characteristic (ROC) Curve')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

--- 
## Summary & Takeaways

1. **Simple vs. Multiple Linear Regression**: Adding the `Sex` categorical feature alongside `Bwt` improved the explanation of variance ($R^2$) in cat heart weight.
2. **Training vs. Testing Performance**: Evaluation on unseen test data ensures that model performance metrics reflect true generalization rather than memorizing training data.
3. **Regression vs. Classification**: Regression models continuous target outcomes (`Hwt`), whereas classification assigns samples to discrete categorical groups (`Sex`).
4. **Metrics Selection**: Choosing appropriate evaluation metrics (e.g., $R^2$/RMSE for regression, F1-Score/ROC-AUC for classification) is crucial for accurate model assessment.